# State-derived adiabatic projector pump

This notebook launches and monitors the S10 static spectral-flow diagnostic. It does **not** evolve the monitored circuit again. Each verified burn-in occupied frame defines its own flattened parent $h_\xi=\mathbf{1}-2P_\xi$, and the fixed-rank occupied projector is continued by overlap through one flux quantum.

## Theory and estimators

For a saved orthonormal occupied frame $F_\xi$, we form $P_\xi=F_\xi F_\xi^\dagger$ and $h_\xi=\mathbf{1}-2P_\xi$. A minimum-image Peierls phase inserts flux along the periodic $y$ direction. At each neighboring flux point the rank-$r$ subspace with greatest overlap with the previous occupied subspace is retained. The charge-transfer estimator is $q_x=(\Delta N_R-\Delta N_L)/2$. Independently filling the lowest $r$ eigenvectors at each flux is saved as the instantaneous closure control.

## Runtime and editable launch configuration

The calculation runs directly from this notebook. One process handles one trajectory/direction path and each process uses one BLAS thread.

In [ ]:
from pathlib import Path
import json, os, subprocess

PROJECT_ROOT = Path('/home/abhuiyan/class_A_fermionic_adaptive_circuit/00_WORKSPACE/CURRENT/frozen_record_flux_charge_pilot')
CONFIG = PROJECT_ROOT / 'campaign_config.state_adiabatic_projector_pump_s10_v1.json'
OUTPUT_ROOT = PROJECT_ROOT / 'results/N16x20_state_adiabatic_projector_pump_s10_v1'
CPU_LIST = '56-63'       # editable logical CPU range
WORKERS = 8              # do not exceed the number of selected CPUs
BLAS_THREADS = 1
print(json.dumps({
    'project_root': str(PROJECT_ROOT), 'config': str(CONFIG),
    'output_root': str(OUTPUT_ROOT),
    'cpu_list': CPU_LIST, 'workers': WORKERS, 'blas_threads': BLAS_THREADS,
}, indent=2))

## Verified input and resume inventory

This cell verifies the 20 source burn-in result/completion pairs and every existing state-pump result before reporting progress.

In [ ]:
command = [
    'taskset', '-c', CPU_LIST, 'python', '-u',
    str(PROJECT_ROOT / 'run_state_adiabatic_projector_pump.py'), 'report',
    '--config', str(CONFIG), '--output-root', str(OUTPUT_ROOT),
]
subprocess.run(command, check=True)

## Run or resume directly

This cell blocks until the static calculation finishes and surfaces both `tqdm` bars. Re-running it verifies all result/completion pairs and skips valid paths.

In [ ]:
environment = os.environ.copy()
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
             'BLIS_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    environment[name] = str(BLAS_THREADS)
run_command = [
    'taskset', '-c', CPU_LIST, 'python', '-u',
    str(PROJECT_ROOT / 'run_state_adiabatic_projector_pump.py'), 'run', '--resume',
    '--config', str(CONFIG), '--output-root', str(OUTPUT_ROOT),
    '--workers', str(WORKERS),
]
subprocess.run(run_command, env=environment, check=True)
subprocess.run([
    'taskset', '-c', CPU_LIST.split(',')[0].split('-')[0], 'python', '-u',
    str(PROJECT_ROOT / 'analyze_state_adiabatic_projector_pump.py'),
    '--config', str(CONFIG), '--output-root', str(OUTPUT_ROOT),
], env=environment, check=True)

## Progress

Run this cell whenever you want a checksum-verified completion count and the latest analysis summary.

In [ ]:
subprocess.run(command, check=True)
summary = OUTPUT_ROOT / 'analysis/analysis_summary.json'
if summary.is_file(): print(summary.read_text())

## Continued projector response

Once all 40 paths verify, the analysis writes the ensemble mean with one-standard-deviation trajectory bands. Each figure has a separate display cell for easy editing.

In [ ]:
from IPython.display import Image, display
figure = OUTPUT_ROOT / 'analysis/figures/state_adiabatic_projector_qx.png'
if figure.is_file():
    display(Image(filename=str(figure)))
else:
    print('Pending: analysis runs automatically after 40/40 verified paths.')

## Left/right subsystem charge

In [ ]:
figure = OUTPUT_ROOT / 'analysis/figures/state_adiabatic_projector_regional_charge.png'
if figure.is_file(): display(Image(filename=str(figure)))
else: print('Pending.')

## Endpoint distributions

In [ ]:
figure = OUTPUT_ROOT / 'analysis/figures/state_adiabatic_projector_endpoints.png'
if figure.is_file(): display(Image(filename=str(figure)))
else: print('Pending.')

## Raw summary and numerical diagnostics

In [ ]:
summary = OUTPUT_ROOT / 'analysis/analysis_summary.json'
if summary.is_file():
    payload = json.loads(summary.read_text())
    print(json.dumps(payload, indent=2, sort_keys=True))
else:
    print('Pending.')